In [1]:
#done and updated by LLZ
# on 28/02/2022

from bs4 import BeautifulSoup
import datetime
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from time import sleep
import os

    

In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'SA SAMA' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.3")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - moodys.com\\Desktop\\Regulator\\{regulatorName}" ## to comment for the local environment

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

Running SA SAMA Web Scraping Tool v.1.3


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

		 'profile.default_content_setting_values.automatic_downloads': 1}

chromeOptions.add_experimental_option("prefs",prefs)


driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()

In [4]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={'SA SAMA 3': 'https://www.sama.gov.sa/en-US/LicenseEntities/Pages/LicensedBanks.aspx',
         'SA SAMA 4': 'https://www.sama.gov.sa/en-US/LicenseEntities/Pages/FinanceLicencedEntities.aspx',

         'SA SAMA 6': 'https://www.sama.gov.sa/en-US/LicenseEntities/Pages/Licensed_Payment_Service_Providers_companies.aspx',
         'SA SAMA 7': 'https://www.sama.gov.sa/en-US/LicenseEntities/Pages/Licensed-Money-Exchangers-Centers.aspx'}
#Removed/outdated
#'SA SAMA 1': 'https://www.sama.gov.sa/en-US/License/Pages/SaudiBanks.aspx', 
#'SA SAMA 2': 'https://www.sama.gov.sa/en-US/License/Pages/InternationalBanks.aspx',
#'SA SAMA 5': 'https://www.sama.gov.sa/en-US/LicenseEntities/Pages/Licensed_Companies.aspx',

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate=now.strftime('%Y-%m-%d')


Typology={

       regulatorName + ' 3': 'Banks',
       regulatorName + ' 4': 'Finance',
       regulatorName + ' 6': 'Payments',
       regulatorName + ' 7': 'Licensed Money Exchange Centers',


        }



In [5]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict

In [6]:
# %%

#------------------------------------------------ Begin_Main ----------------------------------------
for reg in regdict:
    print('Working with {}'.format(reg))
    driver.get(regdict[reg])
    sleep(3)
    soup=BeautifulSoup(driver.page_source, 'html.parser')
    pag_div = soup.find('div', {'class': 'footable-pagination-wrapper'})
    if pag_div is None:
        pages = 1
    else:
        pages = int(pag_div.text.strip().split()[-1])
    for page in range(pages):
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        table = soup.find('table', id='tblPermittedFintech')
        headers = table.find('thead').find_all('tr')[-1]
        headers = [ele.text.strip() for ele in headers]
        trs = table.find('tbody').find_all('tr')
        name_ix = False
        id_1 = False
        id_2 = False
        license_status =False
        if 'Entity Name' in headers:
            name_ix = headers.index('Entity Name')
        elif 'Company Name' in headers:
            name_ix = headers.index('Company Name')
        else:
            for i, hea in enumerate(headers):
                if 'name' in hea.lower():
                    name_ix = i
                    break
            else:
                raise Exception()
        if 'Unified Number' in headers:
            id_1 = ['Unified Number', headers.index('Unified Number')]
        elif 'ID Number' in headers:
            id_1 = ['ID Number', headers.index('ID Number')]
        if 'Commercial Register Number' in headers:
            id_2 = ['Commercial Register Number', headers.index('Commercial Register Number')]
        elif 'ID Licensing Number' in headers:
            id_2 = ['Licensing Number', headers.index('Licensing Number')]
        for tr in trs:
            tds = tr.find_all('td')
            sqldict['ListProcessDate'].append(processdate)
            sqldict['Name'].append(tds[name_ix].text.strip())
            url = tds[name_ix].find('a', href=True) 
            if url:
                sqldict['Website'].append(url['href'])
            if id_1:
                sqldict['InternalID_1_type'].append(id_1[0])
                sqldict['InternalID_1'].append(tds[id_1[1]].text.strip())
            if id_2:
                sqldict['InternalID_2_type'].append(id_2[0])
                sqldict['InternalID_2'].append(tds[id_2[1]].text.strip())
            sqldict['Cntry'].append('SA')
            sqldict['RegCtry'].append('SA')
            sqldict['RegCode'].append('SAMA')
            sqldict['ListCode'].append(reg.split()[-1])
            sqldict['RegulationType'].append('Regulated')##depends
            sqldict['ListName'].append(Typology[reg])
            for key in sqldict:
                if len(sqldict['ListProcessDate'])>len(sqldict[key]):
                    sqldict[key].append('')
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        if soup.find('li', {'data-page':str(page+2)}):
            next_button = driver.find_element(By.XPATH, f"//li[@data-page='{page+2}']/a")
        driver.execute_script("arguments[0].click();", next_button)
        sleep(1.5)


Working with SA SAMA 3
Working with SA SAMA 4
Working with SA SAMA 6
Working with SA SAMA 7


In [7]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

df=pd.DataFrame(sqldict)
df.to_excel(writer, 'SQL Ready', index=False)

writer.save()
writer.close()

sleep(3)

driver.quit()
    

C:\Users\wuj1\AppData\Local\Temp\4\ipykernel_9052\1251123280.py:6: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [10]:
df.to_csv('total_.csv')